In [11]:
%pip install Cython
%pip install transformers torch datasets


  Using cached Cython-3.0.11-cp311-cp311-macosx_10_9_x86_64.whl (3.1 MB)

[notice] A new release of pip available: 22.3.1 -> 24.3.1
[notice] To update, run: python3.11 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
  Using cached datasets-3.1.0-py3-none-any.whl (480 kB)
  Using cached pyarrow-18.0.0.tar.gz (1.1 MB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Discarding https://files.pythonhosted.org/packages/ec/41/6bfd027410ba2cc35da4682394fdc4285dc345b1d99f7bd55e96255d0c7d/pyarrow-18.0.0.tar.gz (from https://pypi.org/simple/pyarrow/) (requires-python:>=3.9): Requested pyarrow>=15.0.0 from https://files.pythonhosted.org/packages/ec/41/6bfd027410ba2cc35da4682394fdc4285dc345b1d99f7bd55e96255d0c7d/pyarrow-18.0.0.tar.gz (from datasets) has inconsistent version: expected '18.0.0', but metadata has '0.0.0'
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.

In [ ]:
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
import torch
from sklearn.model_selection import train_test_split
from datasets import Dataset

#load your labeled data
data = pd.read_csv("/Users/siva/Downloads/Football_training_set.csv")

#map labels to integers
label_mapping = {'Racist': 0, 'Implicitly racist': 1, 'Non-racist': 2}
data['label'] = data['label'].map(label_mapping)

#split into train and validation sets
train_df, val_df = train_test_split(data, test_size=0.2, stratify=data['label'], random_state=42)

#convert to Hugging Face Dataset format
train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)

#load the tokenizer
model_name = "Hate-speech-CNERG/dehatebert-mono-english"
tokenizer = AutoTokenizer.from_pretrained(model_name)

#tokenize function
def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True)

#tokenize datasets
train_dataset = train_dataset.map(tokenize_function, batched=True)
val_dataset = val_dataset.map(tokenize_function, batched=True)

train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
val_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

#load the model
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3)

#set training arguments
training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir='./logs',
)

#set the trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
)

#train the model
trainer.train()


In [ ]:
from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score, f1_score

labels_yt = youtube_comments_df['predicted_label'].values  
#labels_red = reddit_comments_df['predicted_label'].values  

#calculate metrics for comments
accuracy_yt = accuracy_score(labels_yt, predictions_yt)
precision_yt = precision_score(labels_yt, predictions_yt, pos_label=1)
recall_yt = recall_score(labels_yt, predictions_yt, pos_label=1)
f1_yt = f1_score(labels_yt, predictions_yt, pos_label=1)
print("Comments Metrics:")
print(f"Accuracy: {accuracy_yt:.2f}")
print(f"Precision: {precision_yt:.2f}")
print(f"Recall: {recall_yt:.2f}")
print(f"F1 Score: {f1_yt:.2f}")
